# import package:
steps:
1. install package

    on bash: 
    
    `conda install feedparser requests whisper`

    `pip install -q -U google-genai`

    python must >3.9

2. [get Gemini API and set as environment variable](https://ai.google.dev/gemini-api/docs/api-key)

In [ ]:
import os
import time
import feedparser
import requests
import whisper
from google import genai
# from google.api_core import exceptions

# Full code:

this project requires podcast rss link and out put a txt file that contains both original transcript from radio and trranslate version by Gemini.

- get_podcast_by_index:
    input: rss_url, index=0 
        podcast link and video index from the newest to the oldest
    output: title, audio_url
        print the title of video and the url for download 
- download_audio:
    input: audio_url, filename="temp_podcast.mp3"
        radio link for download and filename for saving
    output: filename
        the path of saving mp3 file

- transcribe_locally:
    input: audio_path
        the path of saving mp3 file
    output:result['text']
        the context transferring from radio mp3 to text by whisper
- translate_with_gemini:
    input: text
        the context that need to translate
    output: response.text
        the context after translation by gemini
- main:
    run the functions and then save both original version and translate version. clean the mp3 file before close.
    

In [ ]:
# Whisper 模型大小可选: 'tiny', 'base', 'small', 'medium', 'large'
# 'base' 速度最快且适合 12 分钟音频，'small' 准确率更高一些
WHISPER_MODEL_SIZE = "base" 
# ==========================================

def get_podcast_by_index(rss_url, index=0):
    """
    根据索引获取播客：0 是最新，1 是往后推一期，以此类推
    """
    print(f"正在解析 RSS 链接并获取第 {index + 1} 期内容...")
    feed = feedparser.parse(rss_url)
    
    # 检查索引是否超出范围
    if not feed.entries or index >= len(feed.entries):
        raise Exception(f"索引超出范围！该播客源总共有 {len(feed.entries)} 期节目。")

    # 获取指定索引的文章
    target_episode = feed.entries[index]
    title = target_episode.title

    audio_url = None
    for link in target_episode.links:
        if link.rel == 'enclosure' and 'audio' in link.type:
            audio_url = link.href
            break

    if not audio_url:
        raise Exception(f"在第 {index + 1} 期节目中未找到音频下载链接！")

    print(f"找到节目: {title}")
    return title, audio_url


def download_audio(audio_url, filename="temp_podcast.mp3"):
    """下载音频文件到本地"""
    print(f"正在下载音频 (可能需要几分钟，取决于文件大小): {audio_url}")
    response = requests.get(audio_url, stream=True)
    response.raise_for_status()
    
    with open(filename, 'wb') as f:
        # 分块下载以避免内存占用过大
        for chunk in response.iter_content(chunk_size=8192):
            f.write(chunk)
            
    print("✅ 音频下载完成！")
    return filename


def transcribe_locally(audio_path):
    print(f"正在加载 Whisper ({WHISPER_MODEL_SIZE}) 并进行本地识别...")
    # 这里会自动下载模型文件（仅第一次）
    model = whisper.load_model(WHISPER_MODEL_SIZE)
    result = model.transcribe(audio_path,fp16=False, verbose=False)
    print("✅ 语音识别完成。")
    return result['text']

def translate_with_gemini(text):
    print("正在通过 Gemini API 进行中文翻译...")
    model = "gemini-2.5-flash"
    
    # 针对长文本，建议分段或给出明确指令
    prompt = f"""
    You are a professional translator. Please translate the following English podcast transcript into natural, fluent Simplified Chinese.
    Keep the original meaning but make it sound like a native Chinese podcast.

    ---
    Transcript:
    {text}
    ---
    """
    client = genai.Client()
    try:
        response = client.models.generate_content(model=model, contents = prompt)
        return response.text,response.usage_metadata
    except Exception as e:
        print(f"❌ 运行失败: {e}")
    # for attempt in range(3):
    #     try:
    #         response = client.models.generate_content(model=model, contents = prompt)
    #         return response.text
    #     except Exception as e:
    #         print(f"❌ 运行失败: {e}")
        # except exceptions.ResourceExhausted:
        #     print(f"⚠️ 触发限制，等待 30 秒重试 (第 {attempt+1} 次)...")
        #     time.sleep(30)
    return "翻译失败，请手动将英文文本提交至网页版 Gemini。",""


def main():
    rss_url = input("请输入 RSS 链接: ").strip()
    idx_str = input("你想获取第几期？(默认 1): ").strip()
    index = int(idx_str) - 1 if idx_str else 0
    
    audio_file = "temp_podcast.mp3"
    
    try:
        # 1. 获取信息
        title, audio_url = get_podcast_by_index(rss_url, index)
        
        # 2. 下载音频
        audio_file = download_audio(audio_url, audio_file)
        
        # 3. 本地识别 (不消耗 API 额度)
        english_text = transcribe_locally(audio_file)
        
        # 4. 云端翻译 (只传文本，速度极快)
        chinese_translation = translate_with_gemini(english_text)
        
        # 5. 保存结果
        safe_title = "".join([c for c in title if c.isalnum() or c==' ']).strip()
        filename = f"{safe_title}.txt"
        
        with open(filename, "w", encoding="utf-8") as f:
            f.write(f"TITLE: {title}\n")
            f.write("="*30 + "\n")
            f.write("[English Transcript]\n")
            f.write(english_text)
            f.write("\n\n" + "="*30 + "\n")
            f.write("[Chinese Translation]\n")
            f.write(chinese_translation)
            
        print(f"\n🎉 任务圆满完成！结果已保存至: {filename}")

    except Exception as e:
        print(f"❌ 运行失败: {e}")
    finally:
        if os.path.exists(audio_file):
            os.remove(audio_file)

if __name__ == "__main__":
    main()

# translation by gemini only
this is for decreasing the times of run gemini API during debugging. It can also use for text translation only.

step: change the file_path name, file should contain text that need to be translated. run first and sacond block to check gemini response and token usage. run third block for saving the translation at the same file.

In [ ]:
def translate_with_gemini(text):
    print("正在通过 Gemini API 进行中文翻译...")
    model = "gemini-2.5-flash"
    
    # 针对长文本，建议分段或给出明确指令
    prompt = f"""
    You are a professional translator. Please translate the following English podcast transcript into natural, fluent Simplified Chinese.
    Keep the original meaning but make it sound like a native Chinese podcast.

    ---
    Transcript:
    {text}
    ---
    """
    client = genai.Client()
    try:
        response = client.models.generate_content(model=model, contents = prompt)
        return response.text,response.usage_metadata
    except Exception as e:
        print(f"❌ 运行失败: {e}")
    # for attempt in range(3):
    #     try:
    #         response = client.models.generate_content(model=model, contents = prompt)
    #         return response.text
    #     except Exception as e:
    #         print(f"❌ 运行失败: {e}")
        # except exceptions.ResourceExhausted:
        #     print(f"⚠️ 触发限制，等待 30 秒重试 (第 {attempt+1} 次)...")
        #     time.sleep(30)
    return "翻译失败，请手动将英文文本提交至网页版 Gemini。",""

In [ ]:
#directly give english file
file_path = "/Users/ruoyunma/myGitRepo/ML/dslearn/GeminiAPI/nprnews_test.txt"
with open(file_path, "r", encoding="utf-8") as f:
    english_article = f.read()
chinese_translation,usage = translate_with_gemini(english_article)
if usage:
    print("\n--- Token Usage Metadata ---")
    print(f"Prompt Token Count: {usage.prompt_token_count}")
    print(f"Candidates Token Count (Generated Output): {usage.candidates_token_count}")
    print(f"Total Token Count: {usage.total_token_count}")
else:
    print("\nNo usage metadata found in the response. This can happen with streaming or certain error states.")
print(chinese_translation)

In [ ]:
with open(file_path, "a", encoding="utf-8") as f:
    f.write("="*30 + "\n")
    f.write("chinese_translation:\n")
    f.write(chinese_translation)
print("save successful:",file_path)